# Zalo AI Challenge - Datamate & ZAIC Hybrid Inference trên Google Colab

Notebook này được thiết kế để **chạy suy luận trực tiếp trên tập TEST** trên môi trường Google Colab:
- **Clone trực tiếp mã nguồn từ GitHub**: Không cần tải repo lên Google Drive.
- **Mount Google Drive**: Dùng để lấy tập test (nếu có) và **tự động lưu toàn bộ file kết quả (`submission.jsonl`), báo cáo và logs chi tiết** về Google Drive.
- **Tích hợp Xử lý đầu vào & `as_of_day` từ ZAIC_2026**: Hỗ trợ đa định dạng dữ liệu (CSV, Excel, Parquet, SQLite), DuckDB tối ưu và bộ phân tích thời gian tiếng Việt chuyên sâu (`viet_time.py`, `describe_period`).

## 1. Cấu hình & Clone Repo từ GitHub, Mount Google Drive

In [ ]:
import os, sys, subprocess, shutil, getpass
from pathlib import Path
from urllib.parse import urlparse

IN_COLAB = 'google.colab' in sys.modules

# ======================================================================
# 1. THIẾT LẬP REPO GITHUB CỦA BẠN (Thay đổi nếu dùng repo riêng)
# ======================================================================
REPO_URL = "https://github.com/TRUNGNT0/datamate_zaloAI.git"  # <-- URL repo github
BRANCH = "main"
TARGET_DIR = Path("/content/datamate_repo")

if IN_COLAB:
    # Mount Google Drive
    print("▶ Đang kết nối Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    
    # Clone / Pull repo từ GitHub
    print("▶ Đang lấy mã nguồn từ GitHub...")
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")  # Token từ Colab Secrets (nếu repo private)
    except Exception:
        pass

    clone_url = REPO_URL
    if token:
        clone_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")

    if (TARGET_DIR / ".git").is_dir():
        print(f"Thư mục {TARGET_DIR} đã tồn tại. Đang cập nhật code mới nhất...")
        subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=TARGET_DIR, check=True)
    else:
        print(f"Đang clone {REPO_URL} (branch: {BRANCH})...")
        res = subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", clone_url, str(TARGET_DIR)], capture_output=True, text=True)
        if res.returncode != 0:
            if "Authentication failed" in res.stderr or "could not read Username" in res.stderr:
                token = getpass.getpass("Repo private! Hãy nhập GitHub Personal Access Token (PAT): ")
                clone_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
                subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", clone_url, str(TARGET_DIR)], check=True)
            else:
                raise RuntimeError(f"Lỗi clone Git: {res.stderr}")
    ROOT = TARGET_DIR
else:
    ROOT = Path.cwd().resolve()

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
# Kéo file LFS (nếu adapter được lưu qua Git LFS)
subprocess.run(["git", "lfs", "pull"], cwd=ROOT, check=False)
print(f"✓ Thư mục làm việc: {ROOT}")


## 2. Cài đặt các thư viện phụ thuộc & Kiểm tra GPU

In [ ]:
if IN_COLAB:
    print("▶ Đang cài đặt thư viện cần thiết...")
    !pip install -q duckdb pandas numpy openpyxl xlrd pyarrow python-calamine jsonschema python-dotenv matplotlib
    !pip install -q transformers accelerate peft bitsandbytes

import torch
print(f"PyTorch: {torch.__version__} | CUDA khả dụng: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU: {device_name} ({vram:.1f} GB VRAM)")
else:
    print("⚠️ Lưu ý: Đang chạy trên CPU, hãy đổi Runtime sang GPU (T4, L4 hoặc A100) để suy luận nhanh.")

## 3. Thiết lập Thư mục Lưu Kết Quả & Logs trên Google Drive

In [ ]:
from datetime import datetime

# ĐỊNH NGHĨA ĐƯỜNG DẪN LƯU KẾT QUẢ TRÊN GOOGLE DRIVE
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
if IN_COLAB:
    DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/ZaloAI_Outputs") / f"run_{timestamp}"
else:
    DRIVE_OUTPUT_DIR = ROOT / "outputs" / f"run_{timestamp}"

DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(DRIVE_OUTPUT_DIR / "logs").mkdir(parents=True, exist_ok=True)

# Thư mục tạm tốc độ cao trên Colab SSD
LOCAL_RUN_DIR = Path("/content/run_test_tmp" if IN_COLAB else ROOT / "outputs" / "tmp")
LOCAL_RUN_DIR.mkdir(parents=True, exist_ok=True)
(LOCAL_RUN_DIR / "logs").mkdir(parents=True, exist_ok=True)

print(f"✓ Thư mục đích lưu kết quả & logs trên Drive: {DRIVE_OUTPUT_DIR}")
print(f"✓ Thư mục chạy tạm thời: {LOCAL_RUN_DIR}")

def sync_to_drive():
    """Đồng bộ ngay lập tức submission.jsonl và toàn bộ logs về Google Drive."""
    if not IN_COLAB:
        return
    for item in LOCAL_RUN_DIR.rglob("*"):
        if item.is_file() and not item.name.endswith(".tmp") and not item.name.endswith(".wal"):
            rel = item.relative_to(LOCAL_RUN_DIR)
            dest = DRIVE_OUTPUT_DIR / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(item, dest)
    print(f"[Sync] Đã sao lưu kết quả & logs mới nhất về Google Drive: {DRIVE_OUTPUT_DIR}", flush=True)

## 4. Xác định Tập Dữ Liệu Test (Public Test hoặc Private Test)

Tìm kiếm thư mục chứa tập test trên repo hoặc trên Google Drive.

In [ ]:
# Tìm kiếm tập test tại các đường dẫn khả dĩ
CANDIDATE_PATHS = [
    ROOT / "dataset" / "public_test",
    ROOT / "data" / "public_test",
    Path("/content/drive/MyDrive/ZaloAI/public_test"),
    Path("/content/drive/MyDrive/dataset/public_test"),
    ROOT / "dataset" / "test",
]

TEST_DIR = None
for p in CANDIDATE_PATHS:
    if p.is_dir() and (p / "questions.jsonl").is_file():
        TEST_DIR = p
        break

# Hoặc bạn có thể chỉ định trực tiếp đường dẫn tập test tại đây:
# TEST_DIR = Path("/content/drive/MyDrive/duong_dan_tap_test")

if TEST_DIR is None:
    print("⚠️ Không tự động tìm thấy questions.jsonl trong các đường dẫn mặc định.")
    print("Hãy đặt đường dẫn tập test vào biến TEST_DIR bên dưới.")
    TEST_DIR = ROOT / "dataset" / "public_test"
else:
    print(f"✓ Đã tìm thấy tập test tại: {TEST_DIR}")

# Kiểm tra số lượng câu hỏi và cấu trúc as_of_date / as_of_day
import json
q_file = TEST_DIR / "questions.jsonl"
if q_file.is_file():
    questions = [json.loads(line) for line in q_file.read_text(encoding="utf-8-sig").splitlines() if line.strip()]
    print(f"✓ Tổng số câu hỏi test: {len(questions)}")
    sample = questions[0]
    print(f"✓ Mẫu câu hỏi [0]: ID={sample.get('id')}, Biz={sample.get('business_id')}")
    as_of = sample.get('as_of_date') or sample.get('as_of_day')
    print(f"✓ as_of_date / as_of_day: {as_of}")
    print(f"  Nội dung: {sample.get('question')}")

## 5. Khởi Tạo Pipeline với Xử Lý Đầu Vào & `as_of_day` từ ZAIC_2026

Pipeline sử dụng:
- Xử lý đầu vào: Chuẩn hóa dữ liệu thô, nạp vào DuckDB với kiểm tra schema và UDFs (`vn_date`, `vn_date_year`, `vn_norm`).
- `as_of_day`: Bộ phân tích thời gian chuyên sâu `viet_time.py` & `describe_period` để xác định chính xác khoảng thời gian của câu hỏi.

In [ ]:
from datamate.local_pipeline import LocalPipelineConfig, LocalSQLPipeline
from datamate.tools.timeutil import describe_period
import inspect

# Kiểm tra nhanh bộ phân tích thời gian ZAIC
test_hint = describe_period("tháng này doanh thu bao nhiêu", "2026-05-25")
print(f"✓ Phân tích thời gian ZAIC cho 'tháng này': {test_hint}")

# Cấu hình model & pipeline
MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"  # hoặc Qwen/Qwen3.5-9B
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3) if torch.cuda.is_available() else 0.0
hf_quant = "none" if vram_gb >= 40.0 else "auto"  # T4/V100/L4 dùng 4-bit, A100 dùng full/bf16

cfg_kwargs = dict(
    dataset_dir=str(TEST_DIR.parent),
    out_dir=str(LOCAL_RUN_DIR),
    model=MODEL_NAME,
    hf_quant=hf_quant,
    max_model_len=16384,
    sql_tokens=2048,
    manifest_tokens=3072,
    n_samples=3,
    agree_min=2,
    max_retry=2,
    generation_timeout=600.0,
)

sig = inspect.signature(LocalPipelineConfig)
if "ingestion_mode" in sig.parameters:
    cfg_kwargs["ingestion_mode"] = "direct"
if "require_prepared_database" in sig.parameters:
    cfg_kwargs["require_prepared_database"] = False

pipeline_config = LocalPipelineConfig(**cfg_kwargs)
pipeline = LocalSQLPipeline(pipeline_config)
if hasattr(pipeline, "base"):
    pipeline.base.ingestion_mode = "direct"
    pipeline.base.require_prepared_database = False

print("✓ Khởi tạo LocalSQLPipeline thành công!")

## 6. Nạp Dữ Liệu Shop vào DuckDB (Direct / Layered Ingestion)

In [ ]:
from time import perf_counter

split_name = TEST_DIR.name
t0 = perf_counter()
print(f"▶ Đang chuẩn bị dữ liệu đầu vào cho split '{split_name}'...", flush=True)
prep_report = pipeline.prepare_datasets(splits=(split_name,))
print(f"✓ Chuẩn bị dữ liệu xong sau {perf_counter() - t0:.1f}s!")

# Kiểm tra database
validation = pipeline.validate_databases(splits=(split_name,))
print(f"✓ Đã xác minh thành công {len(validation)} databases shop trong DuckDB.")

# Đồng bộ báo cáo chuẩn bị dữ liệu sang Google Drive
sync_to_drive()


## 7. Chạy Suy Luận Toàn Bộ Tập Test & Tự Động Lưu Về Google Drive

In [ ]:
t_start = perf_counter()
print(f"=== BẮT ĐẦU SUY LUẬN TRÊN TẬP {split_name} ===", flush=True)

# Thực hiện dự đoán
if hasattr(pipeline, f"predict_{split_name}"):
    method = getattr(pipeline, f"predict_{split_name}")
    submission_path = method()
else:
    # Fallback dự đoán tổng quát
    submission_path = pipeline.predict_public_test()

elapsed = perf_counter() - t_start
print(f"\n✓ Hoàn tất suy luận tập test sau {elapsed / 60:.1f} phút!")
print(f"✓ File submission tại: {submission_path}")

# ĐỒNG BỘ TOÀN BỘ KẾT QUẢ VÀ LOGS VỀ GOOGLE DRIVE
sync_to_drive()

# Tải trực tiếp file submission về máy tính cá nhân (nếu muốn)
if IN_COLAB:
    from google.colab import files
    drive_sub = DRIVE_OUTPUT_DIR / split_name / "submission.jsonl"
    target_file = drive_sub if drive_sub.exists() else submission_path
    print(f"▶ Đang tải {target_file} về máy...")
    files.download(str(target_file))

## 8. Kiểm Tra & Thống Kê Kết Quả Đã Lưu Trên Google Drive

In [ ]:
import pandas as pd

print(f"=== DANH SÁCH FILE VÀ LOG ĐÃ LƯU TRONG GOOGLE DRIVE ===\n")
saved_files = sorted(DRIVE_OUTPUT_DIR.rglob("*"))
for f in saved_files:
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        print(f"✓ {f.relative_to(DRIVE_OUTPUT_DIR)} ({size_kb:.1f} KB)")

# Kiểm tra nội dung file submission.jsonl
sub_file = DRIVE_OUTPUT_DIR / split_name / "submission.jsonl"
if not sub_file.exists():
    sub_file = LOCAL_RUN_DIR / split_name / "submission.jsonl"

if sub_file.exists():
    rows = [json.loads(line) for line in sub_file.read_text(encoding="utf-8").splitlines() if line.strip()]
    df_sub = pd.DataFrame(rows)
    print(f"\n✓ Tổng số câu trong submission: {len(df_sub)}")
    print("\nPhân bố loại đáp án dự đoán:")
    print(df_sub['type'].value_counts())
    print("\n10 câu dự đoán đầu tiên:")
    display(df_sub.head(10))
else:
    print("⚠️ Chưa tìm thấy submission.jsonl!")